# BudgiBrain with LangGraph

In [2]:
# Insalling pip dependecies
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
# --- Imports ---
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import datetime
import os
from dotenv import load_dotenv
import re

# --- Setup LLM ---
load_dotenv()
LLM = ChatGroq(
    model_name=os.environ.get("LITELLM_MODEL"),
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

# --- Transaction DB ---
transaction_db = []

# --- API Tools ---
@tool
def add_transaction(amount: Optional[float], category: str, item_name: str, input: str) -> dict:
    """Add a new transaction to the transaction_db."""
    transaction = {
        "datetime": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "amount": amount,
        "category": category,
        "item_name": item_name,
        "input": input
    }
    transaction_db.append(transaction)
    return transaction

@tool
def edit_transaction(category: str, item_name: str, amount: Optional[float] = None) -> dict:
    """Edit the first matching transaction by category and item_name. Only provided fields are updated."""
    for t in transaction_db:
        if (t['category'] == category) and (t['item_name'] == item_name):
            if amount is not None:
                t['amount'] = amount
            return t
    return {"error": "No matching transaction found to edit"}

@tool
def search_transaction_by_category(category: str) -> List[dict]:
    """Search for transactions by category."""
    results = []
    for t in transaction_db:
        if (t['category'] == category):
            results.append(t)
    return results


tools = [add_transaction, edit_transaction, search_transaction_by_category]

# --- State Object ---
class TransactionState(BaseModel):
    input: str = ""
    amount: Optional[float] = None
    category: Optional[str] = None
    item_name: Optional[str] = None
    action: Optional[str] = None  # e.g., "add", "edit", "search"
    result: Optional[dict] = None

# --- Prompts ---
TRANSACTION_PROMPT = """
You are a transaction assistant. Given the user input, extract the following fields as JSON:
- action: one of "add", "edit", "search", "predict"
- amount: the transaction amount (float or null)
- category: the transaction category of Entertainment
- item_name: the item name (string or null)
If a field is missing, set it to null. If action is not clear, guess "add".

Choose the most appropriate category for the given input. The categories are:
- Rent
- Insurance
- Utilities
- Shopping & Entertainment
- Groceries
- Subscriptions
- Transport
- Health

In case you are not sure of the category, prompt the user to provide more information.

User input: {input}
"""

# --- Output Parser ---
class TransactionParse(BaseModel):
    action: str
    amount: Optional[float]
    category: Optional[str]
    item_name: Optional[str]

# --- Node Functions ---
def llm_parse_node(state: TransactionState) -> TransactionState:
    print("➡️ Running llm_parse_node")
    parser = PydanticOutputParser(pydantic_object=TransactionParse)
    prompt = PromptTemplate.from_template(TRANSACTION_PROMPT, partial_variables={"input": state.input})
    chain = prompt | LLM | parser
    parsed: TransactionParse = chain.invoke({"input": state.input})
    print("🧠 Parsed Output:", parsed)
    state.action = parsed.action
    state.amount = parsed.amount
    state.category = parsed.category
    state.item_name = parsed.item_name
    return state

def add_node(state: TransactionState) -> TransactionState:
    result = add_transaction.invoke({
				"amount": state.amount,
				"category": state.category,
				"item_name": state.item_name,
				"input": state.input
		})
    state.result = result
    return state

def edit_node(state: TransactionState) -> TransactionState:
    result = edit_transaction.invoke({
				"category": state.category,
				"item_name": state.item_name,
				"amount": state.amount
		})
    state.result = result
    return state

def search_node(state: TransactionState) -> TransactionState:
    result = search_transaction_by_category.invoke({
				"category": state.category
		})
    state.result = result
    return state


# --- Decision Function ---
def decide_next(state: TransactionState):
    print("🔄 Deciding next step...")
    print("State:", state)
    if state.action == "add" and (state.amount is None or state.category is None or state.item_name is None):
        print("🔁 Re-entering llm_parse (missing add fields)")
        return "llm_parse"
    if state.action == "edit" and (state.category is None or state.item_name is None):
        print("🔁 Re-entering llm_parse (missing edit fields)")
        return "llm_parse"
    if state.action == "add":
        print("✅ Going to add")
        return "add"
    if state.action == "edit":
        print("✅ Going to edit")
        return "edit"
    if state.action == "search":
        print("✅ Going to search")
        return "search"
    # if state.action == "predict":
    #     return "predict"
    print("⏹ Ending graph")
    return END

# --- Build the Graph ---
def build_transaction_graph():
    graph = StateGraph(TransactionState)
    graph.add_node("llm_parse", llm_parse_node)
    graph.add_node("add", add_node)
    graph.add_node("edit", edit_node)
    graph.add_node("search", search_node)
    graph.add_edge("add", END)
    graph.add_edge("edit", END)
    graph.add_edge("search", END)
    graph.add_conditional_edges("llm_parse", decide_next)
    graph.set_entry_point("llm_parse")
    return graph.compile()

transaction_graph = build_transaction_graph()

# --- Example Usage ---
def call_transaction_agent(user_input: str):
    state = TransactionState(input=user_input)
    result = transaction_graph.invoke(state)
    return result["result"]

# --- Cleaned up interactive_chat ---
def interactive_chat():
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    input_box = widgets.Text(
        description='Prompt:',
        placeholder='e.g. Add 500 for groceries as milk',
        layout=widgets.Layout(width='90%')
    )
    output_area = widgets.Output()

    def on_enter(_):
        user_input = input_box.value.strip()
        if user_input:
            result = call_transaction_agent(user_input)
            with output_area:
                clear_output(wait=True)
                print("Result:", result)
            input_box.value = ''

    input_box.on_submit(on_enter)
    display(input_box, output_area)

# To start the chat, just call:
# interactive_chat()

In [4]:
interactive_chat()
# call_transaction_agent("change amount to 1000 on my purchase of milk in groceries")

/var/folders/60/jkpcvndn6j1bchnq17ptwvm80000gn/T/ipykernel_15664/4040138045.py:209: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_enter)


Text(value='', description='Prompt:', layout=Layout(width='90%'), placeholder='e.g. Add 500 for groceries as m…

Output()

In [5]:
print(transaction_db)

[]
